In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

!pip install rasterstats
!pip install rioxarray


In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

import geopandas as gpd
import rioxarray

In [ ]:
def load_terraclimate_dataset(varsagg):
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    ds = ds[varsagg]

    ds = ds.sel(
        time=slice("2010-08-30", "2016-01-02"),
        lat=slice(-21.72, -35.18),
        lon=slice(14.97, 32.79)
    )

   
    return ds

In [ ]:
ds_pre = load_terraclimate_dataset(["ppt", 'aet'])
ds_pre = ds_pre.chunk({'time': 12, 'lat': 1024, 'lon': 1024})
ds_pre.to_zarr("terraclimate_subset.zarr", mode="w")
ds = xr.open_zarr("terraclimate_subset.zarr")

ds = ds.rio.write_crs("EPSG:4326")
basins = gpd.read_file("databases/cuenca09.gpkg")

basins = basins.to_crs("EPSG:4326")

ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
ds = ds.sortby("lat")



In [ ]:
from shapely.geometry import Point

wq_df = pd.read_csv('water_quality_training_dataset.csv')
val_df =  pd.read_csv('submission_template.csv')

gdf_points = gpd.GeoDataFrame(
    wq_df,
    geometry=gpd.points_from_xy(wq_df.Longitude, wq_df.Latitude),
    crs="EPSG:4326"
)
gdf_pointsV = gpd.GeoDataFrame(
    val_df,
    geometry=gpd.points_from_xy(val_df.Longitude, val_df.Latitude),
    crs="EPSG:4326"
)

In [ ]:
train_with_basin = gpd.sjoin(
    gdf_points,
    basins[['HYBAS_ID','geometry']],
    #'AREA_SQKM'
    how="left",
    predicate="intersects"
)

val_with_basin = gpd.sjoin(
    gdf_pointsV,
    basins[['HYBAS_ID','geometry']],
    #'AREA_SQKM',
    how="left",
    predicate="intersects"
)


In [ ]:
def compute_basin_mean(ds, basins_gdf, basin_id, var):

    geom = basins_gdf.loc[
        basins_gdf["HYBAS_ID"] == basin_id,
        "geometry"
    ]

    if geom.empty:
        return None

    try:
        clipped = ds[var].rio.clip(
            geom,
            basins_gdf.crs,
            drop=True
        )

        basin_mean = clipped.mean(dim=["lat","lon"])

        df = basin_mean.to_dataframe().reset_index()
        df["HYBAS_ID"] = basin_id

        return df

    except Exception as e:
        print(f"Error con cuenca {basin_id}: {e}")
        return None



In [ ]:
train_with_basin

In [ ]:
unique_train = train_with_basin["HYBAS_ID"].dropna().unique()

unique_val = val_with_basin["HYBAS_ID"].dropna().unique()

unique_basins = np.unique(np.concatenate([unique_train, unique_val]))
print(len(unique_basins))

In [ ]:
all_basin_stats = []

for bid in tqdm(unique_basins):
    df_basin = compute_basin_mean(ds, basins, bid, "ppt")
    
    if df_basin is not None and not df_basin.empty:
        all_basin_stats.append(df_basin)
print("Cuencas procesadas:", len(all_basin_stats))


In [ ]:
basin_ppt_df = pd.concat(all_basin_stats, ignore_index=True)
basin_ppt_df["time"] = pd.to_datetime(basin_ppt_df["time"])
basin_ppt_df = basin_ppt_df.rename(columns={"ppt": "ppt_mean"})
basin_ppt_df = basin_ppt_df.sort_values(["HYBAS_ID","time"])


In [ ]:
basin_ppt_df["ppt_3m"] = (
    basin_ppt_df
    .groupby("HYBAS_ID")["ppt_mean"]
    .rolling(3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

In [ ]:
basin_ppt_df["ppt_6m"] = (
    basin_ppt_df
    .groupby("HYBAS_ID")["ppt_mean"]
    .rolling(6, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)


In [ ]:
all_basin_stats = []

for bid in tqdm(unique_basins):
    df_basin = compute_basin_mean(ds, basins, bid, "aet")
    
    if df_basin is not None and not df_basin.empty:
        all_basin_stats.append(df_basin)
print("Cuencas procesadas:", len(all_basin_stats))

basin_aet_df = pd.concat(all_basin_stats, ignore_index=True)
basin_aet_df["time"] = pd.to_datetime(basin_aet_df["time"])
basin_aet_df = basin_aet_df.rename(columns={"aet": "aet_mean"})
basin_aet_df = basin_aet_df.sort_values(["HYBAS_ID","time"])

basin_aet_df["aet_3m"] = (
    basin_aet_df
    .groupby("HYBAS_ID")["aet_mean"]
    .rolling(3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)
basin_aet_df["aet_6m"] = (
    basin_aet_df
    .groupby("HYBAS_ID")["aet_mean"]
    .rolling(6, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)


In [ ]:
train_with_basin["Sample Date"] = pd.to_datetime(train_with_basin["Sample Date"], dayfirst=True)
val_with_basin["Sample Date"] = pd.to_datetime(val_with_basin["Sample Date"], dayfirst=True)

train_with_basin["year_month"] = (
    train_with_basin["Sample Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)
val_with_basin["year_month"] = (
    val_with_basin["Sample Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)
basin_ppt_df["year_month"] = (
    basin_ppt_df["time"]
    .dt.to_period("M")
    .dt.to_timestamp()
)
basin_aet_df["year_month"] = (
    basin_aet_df["time"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

train_final = train_with_basin.merge(
    basin_ppt_df,
    left_on=["HYBAS_ID","year_month"],
    right_on=["HYBAS_ID","year_month"],
    how="left"
)
val_final = val_with_basin.merge(
    basin_ppt_df,
    left_on=["HYBAS_ID","year_month"],
    right_on=["HYBAS_ID","year_month"],
    how="left"
)

train_final = train_final.merge(
    basin_aet_df,
    on=["HYBAS_ID","year_month"],
    how="left"
)

val_final = val_final.merge(
    basin_aet_df,
    on=["HYBAS_ID","year_month"],
    how="left"
)



In [ ]:
train_final["water_balance_3m"] = train_final["ppt_3m"] - train_final["aet_3m"]
train_final["water_balance_6m"] = train_final["ppt_6m"] - train_final["aet_6m"]
val_final["water_balance_3m"] = val_final["ppt_3m"] - val_final["aet_3m"]
val_final["water_balance_6m"] = val_final["ppt_6m"] - val_final["aet_6m"]

In [ ]:
train_final = train_final.drop(['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'geometry', 'index_right', 'HYBAS_ID', 'year_month', 'time_x', 'spatial_ref_x', 'crs_x', 'spatial_ref_y', 'crs_y', 'time_y'], axis=1 )
val_final = val_final.drop(['geometry', 'index_right', 'HYBAS_ID', 'year_month', 'time_x', 'spatial_ref_x', 'crs_x', 'time_y', 'spatial_ref_y', 'crs_y'], axis=1 )


In [ ]:
train_final.to_csv("/tmp/hydrobasins_features_train.csv",index = False)

session.sql(f"""
    PUT file:///tmp/hydrobasins_features_train.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()


print("File saved! Refresh the browser to see the files in the sidebar")


val_final.to_csv("/tmp/hydrobasins_features_validation.csv",index = False)

session.sql(f"""
    PUT file:///tmp/hydrobasins_features_validation.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()


print("File saved! Refresh the browser to see the files in the sidebar")

In [ ]:
# funciones helper para cuencas Landsat
import shapely.geometry as geom
from odc.stac import stac_load

import time
from pystac_client.stac_api_io import APIError
from pystac_client import Client
import planetary_computer as pc


def compute_landsat_for_polygon(poly, sample_date):
    """Devuelve medianas de bandas para la geometría y fecha dadas.
    ``poly`` debe ser un objeto shapely (o GeoSeries.iloc[0]) y ``sample_date`` un pd.Timestamp.
    """
    retries=3
    for attempt in range(1, retries+1):
        try:
            catalog = Client.open(
                "https://planetarycomputer.microsoft.com/api/stac/v1",
                modifier=pc.sign_inplace,
            )

    # rango abierto por si no encuentras exactamente el mismo día
    # ensure sample_date is timezone-aware UTC
            if sample_date.tzinfo is None or sample_date.tzinfo.utcoffset(sample_date) is None:
                sample_date = sample_date.tz_localize("UTC")
            else:
                sample_date = sample_date.tz_convert("UTC")

            # Ampliar rango de fechas: ±1 día para encontrar más escenas
            from datetime import timedelta
            dt_start = (sample_date - timedelta(days=1)).strftime("%Y-%m-%d")
            dt_end = (sample_date + timedelta(days=1)).strftime("%Y-%m-%d")
            
            search = catalog.search(
                collections=["landsat-c2-l2"],
                intersects=poly.__geo_interface__,
                datetime=f"{dt_start}/{dt_end}",
                query={"eo:cloud_cover": {"lt": 15}},
            )

            items = search.item_collection()
            if not items:
                return pd.Series({b: np.nan for b in ["blue", "green", "red", "nir08", "swir16", "swir22"]})

            # convert item datetime to UTC and compare
            items = sorted(
                items,
                key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date)
            )
            sel = pc.sign(items[0])

            bands = ["blue","green","red","nir08","swir16","swir22"]
            data = stac_load([sel], bands=bands, geometry=poly, resolution=30).isel(time=0)

            out = {}
            for b in bands:
                arr = data[b].astype("float")
                m = float(arr.median(skipna=True).values)
                out[b] = np.nan if m == 0 else m


            return pd.Series(out)
        except APIError as err:
            if attempt < retries:
                time.sleep(5)           # esperar y volver a intentar
                continue
            else:
                raise   
                

def create_indices(df):
    eps = 1e-10
    band = df.copy()
    band['NDMI'] = (band['nir08'] - band['swir16']) / (band['nir08'] + band['swir16'] + eps)
    band['MNDWI'] = (band['green'] - band['swir16']) / (band['green'] + band['swir16'] + eps)
    band['NDVI'] =(band['nir08']-band['red'])/(band['nir08']+band['red'])
    evi = 2.5 * (band['nir08'] - band['red']) / (band['nir08'] + 6*band['red'] - 7.5*band['blue'] + 1)
    band['EVI'] = evi
    savi = 1.5 * (band['nir08'] - band['red']) / (band['nir08'] + band['red'] + 0.5)
    band['SAVI'] = savi
    nmdi = (band['nir08'] - band['swir16']) / (band['nir08'] + band['swir22'] - band['swir16'] + eps)
    band['NMDI'] = nmdi
    fai = band['nir08'] - (band['red'] + (band['swir16'] - band['red']) * (band['nir08'] - band['red']) / (band['swir16'] - band['red'] + 1e-6))
    band['FAI'] = fai
    turbidity = band['green'] / (band['swir16']+eps)
    band['turbidity'] = turbidity
    ndwi = (band['green'] - band['nir08'])/(band['green'] + band['nir08'])
    band['NDWI'] = ndwi         
    red_green = band['red'] / (band['green']+eps)
    band['red_green'] = red_green                
    swir_nir = band['swir16'] / (band['nir08']+eps)
    band['swir_nir'] = swir_nir                    
    swir2_nir = band['swir22'] / (band['nir08']+eps)
    band['swir2_nir'] = swir2_nir
    band['NDTI'] = (band['red'] - band['green']) / (band['red'] + band['green'] + eps)
    band['BSI'] = ((band['swir16'] + band['red']) - 
               (band['nir08'] + band['blue'])) / (
               (band['swir16'] + band['red']) + 
               (band['nir08'] + band['blue']) + eps)
    band['AWEI'] = 4*(band['green'] - band['swir16']) - \
               (0.25*band['nir08'] + 2.75*band['swir22'])
    band['SI'] = band['swir16'] / (band['green'] + eps)
    return band

    # bucles para procesar los conjuntos de entrenamiento y validación

def process_basins(df, out_path):
    if os.path.exists(out_path):
        print(f"archivo existente {out_path}, cargando y devolviendo")
        return pd.read_csv(out_path)

    results = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        bid = row['HYBAS_ID']
        if pd.isna(bid):
            results.append(pd.Series({b: np.nan for b in ['blue','green','red','nir08','swir16','swir22']}))
            continue
        poly = basins.loc[basins.HYBAS_ID == bid, 'geometry'].iloc[0]
        date = pd.to_datetime(row['Sample Date'], dayfirst=True)
        stats = compute_landsat_for_polygon(poly, date)
        results.append(stats)

    landsat_df = pd.DataFrame(results)
    landsat_df = create_indices(landsat_df)
    landsat_df.to_csv(out_path, index=False)
    return landsat_df

In [ ]:
landsat_train = process_basins(train_with_basin, 'landsat_basins_trai_v2.csv')
landsat_val   = process_basins(val_with_basin,   'landsat_basins_validation_v2.csv')

train_with_basin = pd.concat([train_with_basin.reset_index(drop=True),
                              landsat_train.reset_index(drop=True)], axis=1)

val_with_basin   = pd.concat([val_with_basin.reset_index(drop=True),
                              landsat_val.reset_index(drop=True)], axis=1)

# si quieres agrupar a nivel mensual en lugar de fila a fila, agrupa por year_month aquí


In [ ]:

train_with_basin.to_csv('/tmp/train_with_landsat.csv', index=False)
session.sql(f"""
    PUT file:///tmp/train_with_landsat.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

val_with_basin.to_csv('/tmp/val_with_landsat.csv', index=False)
session.sql(f"""
    PUT file:///tmp/val_with_landsat.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("Landsat datasets saved to Snowflake. Refresh the sidebar to see the files.")
